# Setting a LangChain Chain

In this notebook, we will set a LangChain chain and sent the prompt template to the LLM API to generate a query.

Before getting started, we will connect to the database and set the LLM client.

## Setting the Database Connection

The below code enables us to connect to Postgres (or DuckDB) using the `get_ibis_connection` function:

In [1]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection

Setting a connection to the Postgres database:

In [2]:
tbl_name = "air_traffic"

postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

Or, setting a connection to the DuckDB database:

In [ ]:
# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )

We will use the `ChatPromptTemplate` function to set the prompt template:

In [3]:
from langchain_core.prompts import ChatPromptTemplate

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.
CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)

In [4]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con = con, tbl_name = tbl_name)

schema = tbl_attr.schema

print(schema)

Year bigint, Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint


## Setting the LLM Client

We will use the `ChatOpenAI` to set the LLM client connection:

In [5]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
model = "gpt-4o"

api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
  base_url=base_url, 
  api_key=api_key, 
  temperature=0, model=model
  )


Next, we will set the chain:

In [6]:
chain = prompt_template | llm


In [7]:
print(chain)

first=ChatPromptTemplate(input_variables=['question', 'schema', 'tbl_name'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['schema', 'tbl_name'], input_types={}, partial_variables={}, template="Given the following SQL table, your job is to write queries given a user’s request.\nReturn just the SQL query as plain text, without additional text, and don't use markdown format. \nPlease ensure that the field names in the query are enclosed in double quotes.\nCREATE TABLE {tbl_name} ({schema})"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Write a SQL query that returns: {question}'), additional_kwargs={})]) middle=[] last=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0xfffeee826d50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0xfffeee3

In [8]:
question = "How many passengers landed during 2024?"


Let's now invoke the chain:

In [9]:
llm_output = chain.invoke(
    {
        "question": question,
        "tbl_name": tbl_name,
        "schema": schema,
    }
)


In [10]:
query = llm_output.content
print(query)


SELECT SUM("Passenger Count") AS "Total Passengers Landed"
FROM air_traffic
WHERE "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


In [11]:
con.sql(query).execute()

,Total Passengers Landed
0,26079194


Last but not least, let's functionize this process:

In [12]:
def basic_sql_agent(chain, question, tbl_name, schema, con):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
        }
    )
    query = llm_output.content
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_"* 60)
    output = con.sql(query).execute()
    return output


In [13]:
basic_sql_agent(
    chain,
    question="how many passengers landed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") AS "Total Passengers Landed"
FROM air_traffic
WHERE "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024;
____________________________________________________________


,Total Passengers Landed
0,26079194
